# Leafflication

This projects is an introduction to computer vision.  
It includes image data augmentation, transformation and classification.  

### Imports

In [ ]:
%matplotlib inline

import os
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams["savefig.bbox"] = 'tight'

from torchvision.transforms import v2
from torchvision.io import read_image, write_jpeg


IMGS_NAMES = [
    'Original',
    'RandomCrop',
    'RandomHorizontalFlip',
    'RandomPerspective',
    'RandomRotation',
    'GaussianBlur',
    'ColorJitter',
]

NB_REQUIRED_IMAGES = 1640

# torch.manual_seed(2)

### Data distribution visualization

View image distributions on a pie and a bar chart.  

In [ ]:
APPLES_SUBDIR = './images/Apple/'
GRAPE_SUBDIR = './images/Grape/'

leaves_directory_paths = []
for sub_dir in os.listdir(APPLES_SUBDIR):
    leaves_directory_paths.append(APPLES_SUBDIR + sub_dir)
for sub_dir in os.listdir(GRAPE_SUBDIR):
    leaves_directory_paths.append(GRAPE_SUBDIR + sub_dir)

# is_not_augmented = lambda img_filename : all(sub not in img_filename for sub in IMGS_NAMES)
# get_leaves_imgs = lambda leaves_dir : [img_filename for img_filename in os.listdir(leaves_dir) if is_not_augmented(img_filename)]
get_leaves_imgs = lambda leaves_dir : [img_filename for img_filename in os.listdir(leaves_dir)]

def plot_imgs_distribution():
    # Get the list of subdirectories
    # Count the number of images in each subdirectory
    file_counts = []
    for leaf_directory_path in leaves_directory_paths:
        file_count = len(get_leaves_imgs(leaf_directory_path))
        file_counts.append(file_count)

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    # Pie chart
    ax[0].pie(file_counts, labels=leaves_directory_paths, autopct='%1.1f%%', startangle=140)
    ax[0].axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    ax[0].set_title("Distribution of images (Percentage)")

    # Bar chart
    y_pos = np.arange(len(leaves_directory_paths))
    bars = ax[1].bar(y_pos, file_counts, align='center', alpha=0.7)
    ax[1].set_xticks(y_pos)
    ax[1].set_xticklabels(leaves_directory_paths, rotation=45, ha='right')
    ax[1].set_ylabel('Number of images')
    ax[1].set_title('Distribution of images (Count)')

    # Annotate the bars with their respective counts
    for bar in bars:
        height = bar.get_height()
        ax[1].text(bar.get_x() + bar.get_width()/2., height + 1,
                '%d' % int(height), ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

plot_imgs_distribution()

### Data augmentation
Augment data by cropping, blurring, flipping, rotating, changing colors and applying perspective transformation

In [ ]:
transforms = [
    v2.RandomCrop(230),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomPerspective(distortion_scale=0.4, p=1.0),
    v2.RandomRotation(degrees=(0, 180)),
    v2.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 2.)),
    v2.ColorJitter(brightness=.3, hue=.1),
]

def get_augmentated_images(img_path):
    img = read_image(img_path)
    imgs = [transform(img) for transform in transforms]
    imgs.insert(0, img)
    return imgs

def plot(imgs):
    _, axes = plt.subplots(1, len(imgs), figsize=(15, 5))
    # Convert tensor to numpy array and transpose if necessary
    # Assuming tensors are in the format CxHxW (channels, height, width)
    for i, img in enumerate(imgs):
        transposed_img = img.numpy().transpose(1, 2, 0)
        axes[i].set_title(IMGS_NAMES[i])
        axes[i].imshow(transposed_img)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

def augment_img(path, og_filename, plot_imgs=False):
    imgs = get_augmentated_images(path + '/' + og_filename)
    for i, transformed_img in enumerate(imgs[1:]):
        filename =  path + '/' + og_filename[:-4] + '_' + IMGS_NAMES[i + 1] + '.JPG'
        write_jpeg(input=transformed_img, filename=filename)
    if (plot_imgs):
        plot(imgs)

def augment_leaves_imgs(leaves_dir):
    leaves_imgs_fnames = get_leaves_imgs(leaves_dir)
    nb_og_imgs = len(leaves_imgs_fnames)
    assert nb_og_imgs > 0, "Number of leaves images in " + leaves_dir + " should be at least one."
    nb_augments = (NB_REQUIRED_IMAGES - nb_og_imgs) // len(transforms)
    for augment_i in range(nb_augments):
        img_fname = leaves_imgs_fnames[augment_i % nb_augments]
        augment_img(leaves_dir, img_fname, plot_imgs=(augment_i == nb_augments - 1))

for leaf_directory_path in leaves_directory_paths:
    augment_leaves_imgs(leaf_directory_path)

print('Balanced leaves images to' + NB_REQUIRED_IMAGES + '.')

In [ ]:
plot_imgs_distribution()

### Classification

Classifying the images using a CNN with the following architecture:
- conv layer of 32 8x8 filters
- Max pooling 2x2
- 64 dense layer
- 32 dense layer